# CPSC 4110/5110/7110 — Assignment 4

**Quantum Algorithms, Fall 2026**

**Names:** *Hunter Secretan, Sebastian*

**Covers:** Lectures 10 and 11. Mermin, Chapter 6, and the entanglement swap.

**Due:** Nov 8, 10:00 pm on Moodle. **Total: 60 marks.**

**Teams of two are recommended.** Hand in one notebook per team. Working alone is also fine.
Graduate students have to work individually.

Every question has a simulation part and a part that runs on a **real IBM quantum computer**.

| Question | Topic | Marks |
|:--|:--|--:|
| 1 | Bell states | 12 |
| 2 | The Bell measurement | 12 |
| 3 | Entanglement swapping with a Bell measurement | 14 |
| 4 | A quantum repeater over 8 nodes, using a doubling tree | 22 |

### How to hand this in

1. Work through the notebook in order. Written answers go in the markdown cells marked **Your answer**. Double click to edit.
2. Code answers go in the cells marked `# TODO`. Do not rename the functions.
3. Get everything right on the noisy simulator first (`USE_HARDWARE = False`). Then set `USE_HARDWARE = True` and use `Runtime > Restart and run all` **once**, so the hardware results are in the notebook you submit.
4. Submit the `.ipynb` file. `File > Download > Download .ipynb`.

### Conventions

Throughout, **Qbit 0 is the rightmost**, as in class. Qiskit uses the same ordering, so a counts key `'10'` means Qbit 1 read 1 and Qbit 0 read 0.

**Bell states.** As in Lecture 10, for $x, y \in \{0,1\}$
$$|\beta_{xy}\rangle = \tfrac{1}{\sqrt2}\big(|0\,y\rangle + (-1)^x\,|1\,\bar y\rangle\big), \qquad \bar y = 1 - y .$$
The bit $y$ says whether the two Qbits **agree** ($y=0$) or **differ** ($y=1$). The bit $x$ is the **sign** between the terms.
So $\beta_{00} = \Phi^+$, $\beta_{01} = \Psi^+$, $\beta_{10} = \Phi^-$, $\beta_{11} = \Psi^-$. For two Qbits, the left one is Qbit 1.

In [2]:
%pip install --quiet qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib   
# pylatexenc and matplotlib draw circuits

import numpy as np
from itertools import product
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

sim = AerSimulator()
BASIS = ["cx", "rz", "sx", "x"]

def counts(qc, shots=4096, seed=1234):
    """Run a circuit that ends in measurements on the ideal simulator and return the counts."""
    return sim.run(transpile(qc, sim), shots=shots, seed_simulator=seed).result().get_counts()

def bell_vector(x, y):
    """beta_xy = (|0 y> + (-1)^x |1 ybar>)/sqrt2 as a Statevector. The left Qbit is Qbit 1."""
    v = np.zeros(4, complex); v[2 * 0 + y] += 1; v[2 * 1 + (1 - y)] += (-1) ** x
    return Statevector(v / np.sqrt(2))

def same_state(u, v, tol=1e-9):
    """True if two state vectors are equal up to a global phase."""
    return abs(abs(np.vdot(np.asarray(u), np.asarray(v))) - 1) < tol

def rotate(qc, qubits, basis):
    """Rotate Qbits so that the ordinary measurement measures 'X', 'Y' or 'Z'."""
    for q in qubits:
        if basis == "X": qc.h(q)
        elif basis == "Y": qc.sdg(q); qc.h(q)

def bit(key, k):
    """Classical bit k of a counts key (Qiskit writes bit 0 rightmost)."""
    return int(key.replace(" ", "")[::-1][k])

def correlator(cnts, flip=lambda key: 0, a=0, b=1):
    """<PP> from the classical bits a and b: +1 for even parity, -1 for odd.
    flip(key) = 1 flips the parity of that shot. That is how a correction is applied after the fact."""
    tot = sum(cnts.values())
    return sum(n * (-1) ** (bit(k, a) ^ bit(k, b) ^ flip(k)) for k, n in cnts.items()) / tot

def bell_fidelity(E, x=0, y=0):
    """Fidelity with beta_xy from the correlators E = {'X': <XX>, 'Y': <YY>, 'Z': <ZZ>}.
    For beta_xy the ideal values are <XX> = (-1)^x, <YY> = -(-1)^(x XOR y), <ZZ> = (-1)^y."""
    return (1 + (-1) ** x * E["X"] - (-1) ** (x ^ y) * E["Y"] + (-1) ** y * E["Z"]) / 4

import qiskit, qiskit_aer
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)

Note: you may need to restart the kernel to use updated packages.
qiskit 2.6.2 | aer 0.18.2


---
## Running on the real device

Every question ends with a run on a real IBM quantum computer. **Your IBM Quantum Classroom account is metered: 10 minutes of quantum time per month.**
This whole assignment is 27 circuits; check the estimate after the first hardware job before you continue, and do not run the hardware cells in a loop.

**Setup.** The next two cells work in Colab and on your own machine.

- **In Colab:** open the key icon (Secrets) and add a secret named `IBM_QUANTUM_TOKEN` holding your API key, with notebook access on.
- **On your own machine:** run the save-account cell below once, as its comment says.

**Never type your key into a cell.**

`USE_HARDWARE = False` runs a **noisy simulator of `ibm_fez`** and costs nothing. Get every question right that way first. Then set `USE_HARDWARE = True` and run
the whole notebook once. If a job fails, set `SUBMIT_EACH = True` to see which circuit is the cause. If the notebook closes while a job waits in the queue,
get it back with `service.job("<job id>")` instead of resubmitting.

In [3]:
# ONLY if you run this notebook on your own machine (not in Colab): save your IBM account once.
# DO NOT type your key or instance into this code. The text inside getpass(...) and input(...) is just the
# label of the prompt. Set SAVE_ACCOUNT = True, run the cell, and TYPE the key at the prompt that appears
# (the typing is hidden). It is saved to ~/.qiskit/qiskit-ibm.json and never stored in this notebook.
# Afterwards set SAVE_ACCOUNT back to False.
SAVE_ACCOUNT = False

if SAVE_ACCOUNT:
    from getpass import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService
    token = getpass("Paste your IBM Quantum API key here (hidden), then press Enter: ").strip()
    instance = input("Paste your instance (CRN), or press Enter to skip: ").strip() or None
    if not token:
        raise ValueError("No key entered. Run the cell again and paste your key at the prompt.")
    QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=token, instance=instance,
                                      set_as_default=True, overwrite=True)
    print("Account saved.")

In [4]:
USE_HARDWARE = False     # False: noisy simulator of ibm_fez (free).  True: the REAL device (metered).
BACKEND_NAME = "ibm_fez" # the real device. Set to None to take the least busy one instead.
SUBMIT_EACH = False      # True: one job per circuit, to find out which circuit makes a job fail.

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeFez

try:
    from google.colab import userdata          # running in Colab
    IN_COLAB = True
except ImportError:                            # running on your own machine
    IN_COLAB = False

if USE_HARDWARE:
    if IN_COLAB:
        service = QiskitRuntimeService(channel="ibm_quantum_platform", token=userdata.get("IBM_QUANTUM_TOKEN"))
    else:
        service = QiskitRuntimeService()       # the account you saved with the cell above
    backend = service.backend(BACKEND_NAME) if BACKEND_NAME else service.least_busy(min_num_qubits=14, operational=True, simulator=False)
else:
    backend = FakeFez()                        # a noise model of ibm_fez
    noisy = AerSimulator.from_backend(backend)
print("backend:", backend.name, "| qubits:", backend.num_qubits)

# Scheduling inserts the idle time explicitly, so the simulator also charges for Qbits that wait.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=7, scheduling_method="alap")
RUN_LOG = []

def two_qubit_gates(c):
    return sum(1 for inst in c.data if inst.operation.num_qubits == 2)

def submit(circuits, shots, label):
    """Transpile for the backend and run. Returns (job, transpiled circuits). Every call is recorded in RUN_LOG."""
    isa = [pm.run(c) for c in circuits]
    if SUBMIT_EACH:
        job = []
        for k, c in enumerate(isa):
            if USE_HARDWARE:
                j = SamplerV2(mode=backend).run([c], shots=shots); print(f"{label} circuit {k}: job id", j.job_id())
            else:
                j = noisy.run(c, shots=shots, seed_simulator=11 + k)
            job.append(j)
    elif USE_HARDWARE:
        job = SamplerV2(mode=backend).run(isa, shots=shots); print(f"{label}: job id", job.job_id())
    else:
        job = noisy.run(isa, shots=shots, seed_simulator=11)
    RUN_LOG.append(dict(label=label, job=job, circuits=len(isa), shots=shots))
    return job, isa

def get_counts(job, i):
    """Counts of circuit i. With SUBMIT_EACH = True a failed circuit gives {} and a message."""
    if SUBMIT_EACH:
        try:
            res = job[i].result()
            return res[0].data.c.get_counts() if USE_HARDWARE else res.get_counts()
        except Exception as e:
            print(f"circuit {i} FAILED: {e}"); return {}
    if USE_HARDWARE:
        return job.result()[i].data.c.get_counts()
    return job.result().get_counts(i)

backend: fake_fez | qubits: 156


---
## Question 1. Bell states  *(12 marks)*

The circuit with $H$ on the left Qbit followed by a cNOT (left Qbit controls the right one) sends $|x\,y\rangle$ to $|\beta_{xy}\rangle$.

**(a)** *(3)* By hand, run each of the four inputs $|00\rangle, |01\rangle, |10\rangle, |11\rangle$ through the circuit and write the state after $H$ and after the cNOT.
Say which input bit ends up deciding whether the Qbits agree, and which decides the sign.

**(b)** *(3)* Complete `bell_circuit(x, y)` and check, with `same_state`, that it produces `bell_vector(x, y)` for all four inputs.

**(c)** *(2)* Starting from $\Phi^+$, find a gate on **one** Qbit that gives each of the other three Bell states. Check each one numerically.

**your answer**
**(a)**

$$
\begin{aligned}
|00\rangle
&\xrightarrow{H_1} \frac{1}{\sqrt{2}}\left(|00\rangle+|10\rangle\right)\\
&\xrightarrow{\mathrm{CNOT}_{1\to 0}} \frac{1}{\sqrt{2}}\left(|00\rangle+|11\rangle\right)\\[6pt]
|01\rangle
&\xrightarrow{H_1} \frac{1}{\sqrt{2}}\left(|01\rangle+|11\rangle\right)\\
&\xrightarrow{\mathrm{CNOT}_{1\to 0}} \frac{1}{\sqrt{2}}\left(|01\rangle+|10\rangle\right)\\[6pt]
|10\rangle
&\xrightarrow{H_1} \frac{1}{\sqrt{2}}\left(|00\rangle-|10\rangle\right)\\
&\xrightarrow{\mathrm{CNOT}_{1\to 0}} \frac{1}{\sqrt{2}}\left(|00\rangle-|11\rangle\right)\\[6pt]
|11\rangle
&\xrightarrow{H_1} \frac{1}{\sqrt{2}}\left(|01\rangle-|11\rangle\right)\\
&\xrightarrow{\mathrm{CNOT}_{1\to 0}} \frac{1}{\sqrt{2}}\left(|01\rangle-|10\rangle\right)
\end{aligned}
$$

The input bit $y$ determines whether the qubits agree ($y=0$) or differ ($y=1$).
The input bit $x$ determines the sign: $x=0$ gives $+$ and $x=1$ gives $-$.

In [5]:
# TODO (b): Qbit 1 is the left Qbit. Start in |x y>, apply H to Qbit 1, then a cNOT from Qbit 1 to Qbit 0.

def bell_circuit(x, y):
    qc = QuantumCircuit(2)
    # TODO
    if x == 1:
        qc.x(1)
    if y == 1:
        qc.x(0)
    qc.h(1)
    qc.cx(1, 0)
    return qc

for x, y in product([0, 1], repeat=2):
    print(x, y, same_state(Statevector(bell_circuit(x, y)).data, bell_vector(x, y).data))      # expect True

0 0 True
0 1 True
1 0 True
1 1 True


In [6]:
# TODO (c): apply a one-Qbit gate to Phi+ and compare with the other three Bell states.
from qiskit.circuit.library import XGate, ZGate
phi_plus = Statevector(bell_circuit(0, 0))
print(same_state(Statevector(bell_circuit(0, 0)).data, phi_plus.data)) 

psi_plus = phi_plus.evolve(XGate(), [1])
print(same_state(Statevector(bell_circuit(0, 1)).data, psi_plus.data)) 

phi_minus = phi_plus.evolve(ZGate(), [0])
print(same_state(Statevector(bell_circuit(1, 0)).data, phi_minus.data)) 

psi_minus = phi_plus.evolve(XGate(), [1]).evolve(ZGate(), [0])
print(same_state(Statevector(bell_circuit(1, 1)).data, psi_minus.data))

True
True
True
True


**(d)** *(4)* **Real device.** A Bell state is entangled, so measuring one basis is not enough to check it. Measure the three correlators
$\langle XX\rangle$, $\langle YY\rangle$, $\langle ZZ\rangle$ for each Bell state and estimate its fidelity with `bell_fidelity`.
That is 12 circuits, 1000 shots each. Make a table of the measured fidelities.
Which state comes out worst? Is every one above $\tfrac12$? (A two-Qbit state with fidelity above $\tfrac12$ with a Bell state is certainly entangled.)
Explain in one line why the ideal $\langle ZZ\rangle$ is $(-1)^y$ and the ideal $\langle XX\rangle$ is $(-1)^x$.

**Your answer.**

*(double click to edit)*

In [7]:
def bell_in_basis(x, y, basis):
    """beta_xy, rotated so that measuring both Qbits measures `basis`."""
    qc = QuantumCircuit(2, 2)
    qc.compose(bell_circuit(x, y), inplace=True)
    rotate(qc, [0, 1], basis)
    qc.measure([0, 1], [0, 1])
    return qc

In [8]:
# TODO (d): build the 12 circuits, submit them once with label "Q1", and print the fidelity table.

circuits = [bell_in_basis(x, y, b) for x, y in product([0, 1], repeat=2) for b in "XYZ"]
job1, isa1 = submit(circuits, shots=1000, label="Q1")

# TODO: for each Bell state collect E = {'X': ..., 'Y': ..., 'Z': ...} with correlator(get_counts(job1, i))
# and print its fidelity with bell_fidelity(E, x, y).

for x, y in product([0, 1], repeat=2):
    E = {}
    for j, basis in enumerate("XYZ"):
        i = 3 * (2 * x + y) + j  # circuit index for this state and basis
        E[basis] = correlator(get_counts(job1, i))

    F = bell_fidelity(E, x, y)
    print(f"beta_{x}{y}: E={E}, fidelity={F:.3f}")

beta_00: E={'X': 0.982, 'Y': -0.98, 'Z': 0.966}, fidelity=0.982
beta_01: E={'X': 0.98, 'Y': 0.988, 'Z': -0.978}, fidelity=0.986
beta_10: E={'X': -0.984, 'Y': 0.988, 'Z': 0.98}, fidelity=0.988
beta_11: E={'X': -0.984, 'Y': -0.978, 'Z': -0.984}, fidelity=0.986


---
## Question 2. The Bell measurement  *(12 marks)*

**(a)** *(3)* Run `ordinary_circuit(0, 0)` and `ordinary_circuit(1, 0)` (both Qbits measured in the ordinary way) and compare the counts.
Explain why the ordinary measurement cannot tell $\Phi^+$ from $\Phi^-$.

**(b)** *(3)* A Bell measurement is the preparation circuit run backwards: a cNOT from the first Qbit to the second, then $H$ on the first, then measure.
Complete `bell_measure(qc, first, second)` and check that `bsm_circuit(x, y)` reads `'xy'` with certainty for all four Bell states.

**(c)** *(2)* Two Qbits are in $\Psi^+$. What does the ordinary measurement give, and what does the Bell measurement give? Check by simulation.

**Your answer.**  
$\phi\pm = (\frac{1}{\sqrt 2})(|00\rangle \pm |11\rangle)$  
The two states have different $|11\rangle$ amplitudes, denoted by the sign($\pm$).  

Therefore, by the Born rule, the measurement gives outcome $x$ with probability $|a_x |^2$.

$|00\rangle$ holds probability $|\frac{1}{\sqrt 2}|^2 = \frac{1}{2}$  
$|11\rangle$ holds probability $|\frac{\pm1}{\sqrt 2}|^2 = \frac{1}{2}$  
Normalizing $|00\rangle$ and $|11\rangle$, give us the probability = $1$.  
Therefore, $|01\rangle$ and $|10\rangle$ both hold probability = $0$.  



In [9]:
def ordinary_circuit(x, y):
    """beta_xy, then both Qbits measured in the ordinary way."""
    qc = QuantumCircuit(2, 2)
    qc.compose(bell_circuit(x, y), inplace=True)
    qc.measure([0, 1], [0, 1])
    return qc

print("Phi+ :", counts(ordinary_circuit(0, 0)))
print("Phi- :", counts(ordinary_circuit(1, 0)))

Phi+ : {'11': 2022, '00': 2074}
Phi- : {'11': 2051, '00': 2045}


In [ ]:
# TODO (b): Bell measurement gates. Afterwards, `first` gives x and `second` gives y.

def bell_measure(qc, first, second):
    '''Apply the Bell measurement to Qbits `first` and `second` of qc.'''
    #precaution
    qc.barrier
    #use cNOT
    qc.cx(first, second)
    #use Hadamard
    qc.h(first)
    pass

def bsm_circuit(x, y):
    qc = QuantumCircuit(2, 2)
    qc.compose(bell_circuit(x, y), inplace=True)
    bell_measure(qc, 1, 0)
    qc.measure(1, 1); qc.measure(0, 0)          # the counts key then reads "x y"
    return qc

for x, y in product([0, 1], repeat=2):
    print(f"beta_{x}{y}:", counts(bsm_circuit(x, y)))

beta_00: {'00': 4096}
beta_01: {'01': 4096}
beta_10: {'10': 4096}
beta_11: {'11': 4096}


In [11]:
# TODO (c): Psi+ = beta_01, measured both ways.
print("Psi+ ordinary:", counts(ordinary_circuit(0,1)))
print("Psi+ Bell:    ", counts(bsm_circuit(0,1)))

Psi+ ordinary: {'01': 2022, '10': 2074}
Psi+ Bell:     {'01': 4096}


**(d)** *(4)* **Real device.** Run the four `bsm_circuit` circuits and the two `ordinary_circuit` circuits from (a): 6 circuits, 1000 shots each.
Report, for each Bell state, the probability that the Bell measurement reads its label, and the counts of the two ordinary measurements.
The Bell measurement is only two gates after the preparation. Why is its success probability still below 1? Name the two error sources that matter most.

**Your answer.**

$$

In [ ]:
# TODO (d): submit the 6 circuits once with label "Q2" and print the table.
circuit2 = [bsm_circuit(x, y) for x, y in product([0, 1], repeat=2)] \
         + [ordinary_circuit(0, 1), ordinary_circuit(1, 0)]  #uses Phi- and Psi+ for the ordinary measurement.
job2, isa2 = submit(circuit2, shots=1000, label="Q2")

names = {"00": "Phi+", "01": "Psi+", "10": "Phi-", "11": "Psi-"}

print ("Bell Measurement: Probability of Reading the Label")
for i, (x, y) in enumerate(product([0, 1], repeat=2)):
    c = get_counts(job2, i)         #c = counts
    l = f"{x}{y}"                   #l = label
    t = sum(c.values())             #t = total
    p = c.get(l, 0) / (t or 1)      #p = probability
    print(f"beta_{l} ({names[l]}): \n  P('{l}') = {p:.3f}  \n  counts = {dict(sorted(c.items()))}"
          f"2Q gates = {two_qubit_gates(isa2[i])}")

print ("\nOrdinary Measurement: Probability of Reading the Label")
for i, name in [(4, "Phi+"), (5, "Psi+")]:
    print(f"{name}: \n  counts = {dict(sorted(c.items()))}"
          f"2Q gates = {two_qubit_gates(isa2[i])}")

Bell Measurement: Probability of Reading the Label
beta_00 (Phi+): 
  P('00') = 0.996  
  counts = {'00': 996, '01': 3, '10': 1}2Q gates = 0
beta_01 (Psi+): 
  P('01') = 0.997  
  counts = {'01': 997, '11': 3}2Q gates = 0
beta_10 (Phi-): 
  P('10') = 0.998  
  counts = {'00': 1, '10': 998, '11': 1}2Q gates = 0
beta_11 (Psi-): 
  P('11') = 0.997  
  counts = {'01': 1, '10': 2, '11': 997}2Q gates = 0

Ordinary Measurement: Probability of Reading the Label
Phi+: 
  counts = {'01': 1, '10': 2, '11': 997}2Q gates = 1
Psi+: 
  counts = {'01': 1, '10': 2, '11': 997}2Q gates = 1


---
## Question 3. Entanglement swapping with a Bell measurement  *(14 marks)*

Four Qbits, in the ket order $|A\,B_1\,B_2\,C\rangle$, so $A$ is Qbit 3 and $C$ is Qbit 0. $A$ and $B_1$ share $\Phi^+$, and $B_2$ and $C$ share $\Phi^+$.
Node $B$ holds $B_1$ and $B_2$ and does a Bell measurement on them, with $B_1$ first: it reads $x$ from $B_1$ and $y$ from $B_2$.

**(a)** *(3)* By hand: multiply out $\Phi^+_{AB_1}\,\Phi^+_{B_2C}$ and show that, when $B$ reads $x\,y$, Qbits $A$ and $C$ are left in $\beta_{xy}$, each outcome with
probability $\tfrac14$. Which correction does $C$ apply to get $\Phi^+$?

**(b)** *(4)* Complete `swap_unitary()`: the two Bell pairs, then $B$'s Bell measurement gates. The given `ac_after(x, y)` returns the state of $A$ and $C$
after $B$ reads $x\,y$. Check that it is $\beta_{xy}$ with probability $\tfrac14$ for all four outcomes.

**(c)** *(2)* On the device we do not apply $C$'s correction ($X^y$, then $Z^x$) with gates. Instead we measure $A$ and $C$ and fix each shot afterwards:
the $ZZ$ parity is flipped when $y = 1$, the $XX$ parity when $x = 1$, and the $YY$ parity when $x \oplus y = 1$. Explain why, and complete `swap_flip(basis)`.

answer **(a)** The initial state is two $\Phi^+$ pairs. In the qubit order $A,B_1,B_2,C$:

$$
\begin{aligned}
|\Phi^+\rangle_{AB_1}|\Phi^+\rangle_{B_2C}
= \frac{1}{2}\bigl(&|0000\rangle + |0011\rangle\\
                   &+ |1100\rangle + |1111\rangle\bigr).
\end{aligned}
$$

Regrouping the same state by the outer qubits $A,C$ and the middle qubits $B_1,B_2$ gives:

$$
\begin{aligned}
|\Phi^+\rangle_{AB_1}|\Phi^+\rangle_{B_2C}
= \frac{1}{2}\bigl(&|\beta_{00}\rangle_{AC}|\beta_{00}\rangle_{B_1B_2}\\
                   &+ |\beta_{01}\rangle_{AC}|\beta_{01}\rangle_{B_1B_2}\\
                   &+ |\beta_{10}\rangle_{AC}|\beta_{10}\rangle_{B_1B_2}\\
                   &+ |\beta_{11}\rangle_{AC}|\beta_{11}\rangle_{B_1B_2}\bigr).
\end{aligned}
$$

B measures the middle qubits, $B_1B_2$, in the Bell basis. If the result is $xy$, then $A,C$ are left in $|\beta_{xy}\rangle$. Each result has amplitude $1/2$, so its probability is $(1/2)^2=1/4$.

To turn $|\beta_{xy}\rangle_{AC}$ into $|\Phi^+\rangle$, C applies $X$ if $y=1$, then $Z$ if $x=1$.

In [20]:
A, B1, B2, C = 3, 2, 1, 0        # Qbit numbers; the ket order is |A B1 B2 C>

def ac_after(x, y):
    """State of (A, C), A on the left, after B reads x from B1 and y from B2. Returns (state, probability)."""
    T = Statevector(swap_unitary()).data.reshape(2, 2, 2, 2)     # axes: A, B1, B2, C
    v = T[:, x, y, :].reshape(4)
    n = np.linalg.norm(v)
    if n < 1e-12:                    # outcome x y cannot happen
        return v, 0.0
    return v / n, n ** 2

In [21]:
# TODO (b): Phi+ on (A, B1) and on (B2, C), then B's Bell measurement gates on (B1, B2).

def swap_unitary():
    qc = QuantumCircuit(4)
    # TODO
    #prepare phi+ between A and B1
    qc.h(A)
    qc.cx(A, B1)
    #prepare phi+ between B2 and C
    qc.h(B2)
    qc.cx(B2, C)
    #Bell measurement on B1 and B2
    qc.cx(B1, B2)
    qc.h(B1)
    
    return qc

for x, y in product([0, 1], repeat=2):
    v, p = ac_after(x, y)
    print(f"B reads {x}{y}: probability {p:.3f}, A and C in beta_{x}{y}: {same_state(v, bell_vector(x, y).data)}")

B reads 00: probability 0.250, A and C in beta_00: True
B reads 01: probability 0.250, A and C in beta_01: True
B reads 10: probability 0.250, A and C in beta_10: True
B reads 11: probability 0.250, A and C in beta_11: True


In [22]:
# TODO (c): the parity flip for each basis. key is a counts key with bits: 0 = x, 1 = y, 2 = A, 3 = C.

def swap_flip(basis):
    # TODO: return a function key -> 0 or 1
    if basis == "X":
        return lambda key: bit(key, 0) #flip parity if x = 1
    elif basis == "Y":
        return lambda key: bit(key, 0) ^ bit(key, 1) #if x xor y = 1
    elif basis == "Z":
        return lambda key: bit(key, 1) #flip parity if y = 1
    else:
        raise ValueError(f"Unknown basis {basis}")
    return lambda key: 0

def swap_in_basis(basis):
    qc = QuantumCircuit(4, 4)
    qc.compose(swap_unitary(), inplace=True)
    rotate(qc, [A, C], basis)
    qc.measure(B1, 0); qc.measure(B2, 1); qc.measure(A, 2); qc.measure(C, 3)
    return qc

E = {b: correlator(counts(swap_in_basis(b)), flip=swap_flip(b), a=2, b=3) for b in "XYZ"}
print(E, " fidelity with Phi+:", bell_fidelity(E))       # expect 1.0 on the ideal simulator

{'X': 1.0, 'Y': -1.0, 'Z': 1.0}  fidelity with Phi+: 1.0


**(d)** *(5)* **Real device.** Run `swap_in_basis` for X, Y and Z (3 circuits, 4000 shots each), apply the corrections after the fact, and compute the
fidelity of the $A$–$C$ pair with $\Phi^+$. Compare it with the directly prepared $\Phi^+$ from Question 1. Are $A$ and $C$ still entangled?
Count the two-Qbit gates in the transpiled circuit and say what costs the fidelity.

**Your answer.**

*(double click to edit)*

In [ ]:
circuits3 = [swap_in_basis(basis) for basis in "XYZ"]
job3, isa3 = submit(circuits3, shots=4000, label="Q3")

E_swap = {
    basis: correlator(
        get_counts(job3, i),
        flip=swap_flip(basis),
        a=2,
        b=3
    )
    for i, basis in enumerate("XYZ")
}

F_swap = bell_fidelity(E_swap)
print("Corrected correlators:", E_swap)
print("Entanglement-swapping fidelity:", F_swap)

for basis, circuit in zip("XYZ", isa3):
    print(f"{basis} circuit two-qubit gates:", two_qubit_gates(circuit))

---
## Question 4. A quantum repeater over 8 nodes, using a doubling tree  *(22 marks)*

Eight nodes $0, 1, \dots, 7$ stand in a line. Neighbours share a short link, a $\Phi^+$ pair, so there are seven links. Node $i$ holds Qbit $L(i)$ for the link to
its left and Qbit $R(i)$ for the link to its right, with $R(i) = 2i$ and $L(i) = 2i - 1$. That is 14 Qbits. A **swap at node $i$** is a Bell measurement on
$(L(i), R(i))$, with $L(i)$ first: it reads $x_i$ from $L(i)$ and $y_i$ from $R(i)$. The goal is a $\Phi^+$ pair between $R(0)$ at node 0 and $L(7)$ at node 7.

The **doubling tree** swaps in rounds: round 1 at nodes 1, 3, 5; round 2 at nodes 2 and 6; round 3 at node 4. The **chain** swaps one node per round: 1, 2, 3, 4, 5, 6.

**(a)** *(4)* List the links that exist after each round of the tree. How many swaps, how many classical bits and how many rounds does the tree need? And the chain?
Roughly how many rounds does a doubling tree need for $n$ nodes?

**(b)** *(4)* **The composition law.** If $A$–$B_1$ is in $\beta_{ab}$ and $B_2$–$C$ is in $\beta_{cd}$, and $B$ reads $x\,y$, then $A$–$C$ is left in $\beta_{a \oplus c \oplus x,\; b \oplus d \oplus y}$.
Check this by simulation for all 64 cases. Then explain why it means no node has to correct anything during the repeater: the end pair is
$\beta_{XY}$ with $X = \bigoplus_i x_i$ and $Y = \bigoplus_i y_i$, and one correction at the end is enough.

**Your answer.**

*(double click to edit)*

In [ ]:
# TODO (b): prepare beta_ab on (A, B1) and beta_cd on (B2, C), apply B's Bell measurement gates, project, compare.
# Hint: start from swap_unitary(), but replace the two Bell pairs with bell_circuit(a, b) on (A, B1) and bell_circuit(c, d) on (B2, C).

failures = 0
# TODO: loop over a, b, c, d, x, y and count the cases where A-C is not beta_{a^c^x, b^d^y}
print("failures:", failures, "of 64")

**(c)** *(5)* Complete `repeater_circuit(rounds, basis)`: make the seven $\Phi^+$ pairs, put a `barrier()` after them and after every round (a barrier is one
time step), do the swaps of each round, rotate the two end Qbits to `basis`, and measure. The bit layout is fixed: node $i$'s $x_i$ goes to bit $2(i-1)$ and $y_i$ to
bit $2(i-1)+1$; $R(0)$ goes to bit 12 and $L(7)$ to bit 13. Complete `repeater_flip(basis)` from the composition law, and check that both the tree and the chain give
fidelity 1 on the ideal simulator.

**(d)** *(3)* Transpile both schedules with `pm` and report the two-Qbit gate count, the depth and the duration (`circuit.estimate_duration(backend.target, unit="u")`,
in microseconds). Why are the gate counts equal while the durations differ? Which Qbits are waiting during the extra time?

**Your answer.**

*(double click to edit)*

In [ ]:
N = 8
def R(i): return 2 * i            # node i's Qbit for the link to its right   (nodes 0..6)
def L(i): return 2 * i - 1        # node i's Qbit for the link to its left    (nodes 1..7)
TREE = [[1, 3, 5], [2, 6], [4]]
CHAIN = [[1], [2], [3], [4], [5], [6]]

# TODO (c)
def repeater_circuit(rounds, basis):
    qc = QuantumCircuit(2 * (N - 1), 2 * (N - 1))
    # TODO: seven Phi+ pairs, barrier; each round: Bell measurements at its nodes, barrier; rotate the ends; measure
    return qc

def repeater_flip(basis):
    # TODO: use X = XOR of all x_i (even bits 0..10) and Y = XOR of all y_i (odd bits 1..11)
    return lambda key: 0

for name, rounds in (("tree", TREE), ("chain", CHAIN)):
    E = {b: correlator(counts(repeater_circuit(rounds, b)), flip=repeater_flip(b), a=12, b=13) for b in "XYZ"}
    print(name, "fidelity with Phi+:", bell_fidelity(E))

In [ ]:
# TODO (d): pm.run each schedule's Z circuit and print two_qubit_gates, depth and duration.

**(e)** *(6)* **Real device.** Run the tree and the chain in X, Y and Z: 6 circuits of 14 Qbits, 4000 shots each. Compute the end-to-end fidelity for each schedule.
Put it next to Question 1 (no swap) and Question 3 (one swap): how does the fidelity fall with the number of swaps? Is the 8-node pair still entangled ($F > \tfrac12$)?
Does the tree beat the chain on the device, and by about as much as the noisy simulator predicted?
Finally, in a few lines: why do real repeaters use a doubling tree when links only succeed with some probability, and what must they add to go much further?

**Your answer.**

*(double click to edit)*

In [ ]:
# TODO (e): submit the 6 circuits with label "Q4", compute F for the tree and the chain, and print a table with
# Question 1 (0 swaps), Question 3 (1 swap) and the two repeaters (6 swaps).

---
## Run log

In [ ]:
# Run log. Run this AFTER your hardware run and copy what it prints into the Run log cell below.
from datetime import datetime
print("date    :", datetime.now().strftime("%Y-%m-%d %H:%M"), "| backend:", backend.name, "| hardware:", USE_HARDWARE)
total = 0.0
for r in RUN_LOG:
    jobs = r["job"] if isinstance(r["job"], list) else [r["job"]]
    if USE_HARDWARE:
        used = sum(j.usage() for j in jobs); total += used
        print(f"{r['label']}: {r['circuits']} circuits x {r['shots']} shots, job id(s) {[j.job_id() for j in jobs]}, quantum time {used} s")
    else:
        print(f"{r['label']}: {r['circuits']} circuits x {r['shots']} shots (simulator, no quantum time)")
if USE_HARDWARE:
    print("total quantum time (s):", total, " (reads 0 until IBM finishes its accounting; run this cell again in a minute)")

---
**Run log.** Backend, date, job ids and quantum time for your hardware run, copied from the cell above.

*(double click to edit)*

---
### Before you submit

- Every **Your answer** cell filled in, and every `# TODO` replaced.
- One hardware run with `USE_HARDWARE = True`, and its outputs kept. The **Run log** is filled in.
- Your IBM API key is **not** anywhere in the notebook.
- Both names at the top.
- Name anyone outside your team you discussed the problems with. See the course outline for the AI
  policy: use it to develop your understanding, not to produce the answers you hand in.